# 04 - Submission And Report

Goal: validate the exported packages, create the final `TEAMNAME_AGENT.zip`, and gather report-ready plots/evaluation tables.

In [ ]:
from pathlib import Path
import sys

def _add_project_root_to_path():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "soccer-twos-starter"):
            if (candidate / "soccer_twos_project" / "notebook_tools.py").exists():
                if str(candidate) not in sys.path:
                    sys.path.insert(0, str(candidate))
                return candidate
    raise FileNotFoundError("Could not find the soccer-twos-starter project root.")

_add_project_root_to_path()

import importlib
import soccer_twos_project.notebook_tools as notebook_tools
importlib.reload(notebook_tools)
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

## Assignment Requirements Checklist

Updated PDF requirements: submit one zip containing `TEAMNAME_AGENT/`; include `__init__.py`, an `AgentInterface` implementation, runtime model files, and README metadata. Report needs learning curves for every discussed agent and direct comparison plots.

In [ ]:
artifact_checklist(ctx)

## Validate Exported Packages

Each package must import, instantiate with a real SoccerTwos env, and return actions for both players.

In [ ]:
for module_name in ["soccer_ppo_baseline", "soccer_ppo_shaped", "soccer_ppo_curriculum", "soccer_bc_imitation"]:
    try:
        validate_agent_package(ctx, module_name)
    except Exception as exc:
        print(module_name, "FAILED:", exc)

## Final Evaluations

Run quick 5-episode checks first. Use 100 episodes for report numbers once packages work. If the TA agent is released, evaluate the final candidate against it too.

In [ ]:
from soccer_twos_project.evaluation import evaluate, safe_label, write_outputs

def evaluate_pair(agent1, agent2, episodes=5):
    rows, summary = evaluate(agent1, agent2, episodes=episodes, base_port=None)
    write_outputs(rows, summary, ctx.dirs["evals"], safe_label(agent1, agent2))
    print_json(summary)
    return summary

# Quick checks:
# evaluate_pair("soccer_ppo_curriculum", "soccer_ppo_baseline", episodes=5)
# evaluate_pair("soccer_ppo_curriculum", "ceia_baseline_agent", episodes=5)

# Final report runs:
# evaluate_pair("soccer_ppo_baseline", "ceia_baseline_agent", episodes=100)
# evaluate_pair("soccer_ppo_shaped", "ceia_baseline_agent", episodes=100)
# evaluate_pair("soccer_ppo_curriculum", "ceia_baseline_agent", episodes=100)
# evaluate_pair("soccer_bc_imitation", "ceia_baseline_agent", episodes=100)

## Evaluation Table For Report

In [ ]:
evaluation_summaries(ctx)

## Rebuild Plots

The report needs one training curve per discussed agent and one direct comparison plot.

In [ ]:
from soccer_twos_project.plotting import plot_results

plot_results(SimpleNamespace(
    artifact_root=str(ctx.artifact_root),
    ray_results=str(ctx.dirs["checkpoints"]),
    output_dir=str(ctx.dirs["plots"]),
    filter=None,
))

## Create Final Submission Zip

Choose the strongest validated agent. The zip must extract to exactly one top-level folder named `TEAMNAME_AGENT`.

In [ ]:
SOURCE_AGENT = "soccer_ppo_curriculum"  # change to soccer_ppo_selfplay if that is stronger
TEAM_AGENT_NAME = "TEAMNAME_AGENT"  # replace TEAMNAME before final export

final_zip = make_final_submission(ctx, SOURCE_AGENT, TEAM_AGENT_NAME)
validate_zip_package(final_zip)
final_zip

## Report Skeleton

Use the CoRL template. Keep it concise: abstract, one-paragraph overview, method, preliminary results, analysis. Include exact hyperparameters from each run's `run_metadata.json` and the generated plots/evaluation tables.

In [ ]:
for stage in ["ppo_baseline", "ppo_shaped", "ppo_curriculum"]:
    print("\n==", stage, "==")
    run_metadata_status(ctx, stage)